# mcp

> Route stdio MCP requests to rustygate gateways

In [ ]:
#| default_exp mcp

Claude Code starts `clikernel-mcp` for each conversation. Its `Router` accepts MCP messages over stdio and sends tool calls to rustygate gateways. Rustygate implements the tools. The router gets their schemas from the local gateway and adds `host` to `list_kernels`, `use_kernel`, and `create`.

Each host has a separate gateway session. The router remembers one current host for calls without an explicit `host`. It removes `host` before forwarding a call but preserves the JSON-RPC request id. Cancellation notifications also retain their original ids.

`main` runs the router and closes its gateway sessions on exit. Closing a session sends an HTTP DELETE. The gateway then stops the kernels that session created with autoclose. If the router started a child gateway, it stops that process too.


In [ ]:
#| export
import asyncio
from fastcore.utils import *
from fastcore.script import call_parse, store_true
from mcpmini.core import serve_stdio, jresp, jerr
from clikernel.core import Gateway, default_gateway, resolve, session_defaults
from clikernel import __version__


In [ ]:
from fastcore.test import *
from itertools import count
from IPython.display import Markdown
import re, tempfile
from mcpmini.core import jreq, jtool
from rustygate.tools import start_gateway

## Gateway sessions

In [ ]:
#| export
HOST_PARAM = {'type': 'string', 'description': 'Gateway to target: a gateways.toml name, or empty for the default local gateway'}
HOSTED = ('list_kernels', 'use_kernel', 'create')

class Router:
    "Route stdio MCP to rustygate with one session per host and one current host."
    def __init__(
        self,
        cfgdir=None,  # Config dir for `session_defaults` and `gateways.toml` (the standard one if None)
        quiet=False,  # Keep startup output out of replies?
    ):
        self.cfgdir,self.quiet,self.sessions,self.cur,self.child = cfgdir,quiet,{},'',None

    async def session(self, host=''):
        "Return the session for `host`, initializing it on first use. Empty `host` selects the default local gateway."
        if host not in self.sessions:
            if host:
                url, token, verify = resolve(host, self.cfgdir)
                self.sessions[host] = await Gateway(url, token, verify).initialize(session_defaults(self.cfgdir, self.quiet, local=False))
            else: self.sessions[host], self.child = await default_gateway(self.cfgdir, self.quiet)
        return self.sessions[host]

The examples start a disposable gateway on a free local port. A temporary `startup.py` defines `base = 42` and prints `ready`. `session()` opens the default local gateway session on first use. The gateway supplies its kernel-selection instructions during initialization.


In [ ]:
g = start_gateway()
os.environ['CLIKERNEL_HOST'] = g.url
tmp = tempfile.TemporaryDirectory()
cfgd = Path(tmp.name)
(cfgd/'startup.py').write_text('base = 42; print("ready")')
router = Router(cfgd)
local = await router.session()
test_eq(local.url, g.url)
local.info['instructions']

"Select a kernel with create(kernel=...) or use_kernel, then run code with exec(code=...). Kernel implementations are listed in create's schema; creation and selection return usage guidance. New kernels require an explicit implementation; exec never creates or switches kernels. An optional dlgname binds creation to a dialog or overrides execution for one call. Kernels this session creates autoclose by default. Python startup/inspectors run only in Python kernels."

`tools` takes the gateway's schemas and adds `host` to `list_kernels`, `use_kernel`, and `create`. Execution follows the current host instead of accepting a host of its own.

In [ ]:
#| export
@patch
async def tools(self:Router):
    "Return the local gateway's tools with `host` added to `list_kernels`, `use_kernel`, and `create`."
    ts = await (await self.session()).tools()
    for t in ts:
        if t['name'] in HOSTED: t['inputSchema'].setdefault('properties', {})['host'] = dict(HOST_PARAM)
    return ts

These are the tools whose schemas let the client choose a host:

In [ ]:
listed = await router.tools()
byname = {t['name']: t for t in listed}
assert all('host' in byname[n]['inputSchema']['properties'] for n in HOSTED)
assert 'host' not in byname['exec']['inputSchema']['properties']
[n for n,t in byname.items() if 'host' in t['inputSchema']['properties']]


['list_kernels', 'create', 'use_kernel']

## Dispatching requests

`dispatch` answers `initialize` and `ping` locally. It forwards tool calls with their original JSON-RPC ids. `create` and `use_kernel` change the current host when given an explicit `host`; `list_kernels` does not.

Cancellation notifications go to the current gateway with their original request ids. The router does not track the host of each outstanding request. Switching hosts before cancellation can send the notification to a different gateway.

In [ ]:
#| export
@patch
async def dispatch(self:Router, msg, requester=None):
    "Answer initialization and ping locally. Forward tool calls to gateways."
    method,id = msg.get('method'), msg.get('id')
    try:
        if method == 'initialize':
            info = (await self.session()).info
            return jresp(id, dict(protocolVersion=msg['params'].get('protocolVersion', '2025-06-18'), capabilities=dict(tools={}),
                serverInfo=dict(name='clikernel', version=__version__), instructions=info.get('instructions')))
        if id is None:
            if method == 'notifications/cancelled': await (await self.session(self.cur)).tr.send(msg)
            return None
        if method == 'ping': return jresp(id, {})
        if method == 'tools/list': return jresp(id, dict(tools=await self.tools()))
        if method == 'tools/call':
            args = msg['params'].setdefault('arguments', {})
            has_host = 'host' in args
            host = args.pop('host', '') or ''
            s = await self.session(host if has_host else self.cur)
            if has_host and msg['params']['name'] in ('use_kernel', 'create'): self.cur = host
            return await s.tr.send(msg)
        return jerr(id, -32601, f'method not found: {method}')
    except Exception as e: return None if id is None else jerr(id, -32603, str(e))

The stdio client initializes its own session with the router. The reply uses the client's protocol version and includes the local gateway's instructions:

In [ ]:
init = await router.dispatch(jreq('initialize', 1, protocolVersion='2025-11-25', capabilities={}, clientInfo=dict(name='demo', version='0')))
test_eq(init['result']['serverInfo']['name'], 'clikernel')
test_eq(init['result']['protocolVersion'], '2025-11-25')
Markdown(init['result']['instructions'])

Select a kernel with create(kernel=...) or use_kernel, then run code with exec(code=...). Kernel implementations are listed in create's schema; creation and selection return usage guidance. New kernels require an explicit implementation; exec never creates or switches kernels. An optional dlgname binds creation to a dialog or overrides execution for one call. Kernels this session creates autoclose by default. Python startup/inspectors run only in Python kernels.

The remaining lessons use `call` for an MCP tool result and `text` for its text content. These notebook helpers assign request ids and remove the JSON-RPC envelope.

In [ ]:
#| hide
request_ids = count(2)
def txt(r): return ''.join(c.get('text', '') for c in r['content'] if c['type'] == 'text')
async def call(name, **args): return (await router.dispatch(jtool(name, next(request_ids), **args)))['result']
async def text(name, **args):
    r = await call(name, **args)
    assert not r.get('isError'), txt(r)
    return txt(r)

Creating a Python kernel runs `startup.py`. The next execution sees the value it defined:

In [ ]:
created = await text('create', kernel='py')
assert 'created kernel' in created and 'ready' in created
base = await text('exec', code='base')
test_eq(base, '42')
base

'42'

`gateways.toml` maps host names to gateways. Here, `alt` refers to a second disposable gateway. Creating a kernel there makes it the router's current host. The following Python calls omit `host` and run in that kernel:

In [ ]:
g2 = start_gateway()
(cfgd/'gateways.toml').write_text(f'[gateways.alt]\nurl = "{g2.url}"\n')
created = await text('create', dlgname='far.ipynb', kernel='py', host='alt')
assert 'created kernel' in created and 'ready' in created
await text('exec', code='marker = 7')
marker = await text('exec', code='marker')
test_eq(marker, '7')
marker

'7'

Use `host=''` to address the default local gateway. Listing its kernels doesn't change the current host. Selecting its kernel with `use_kernel` does.

The local kernel has `base` from startup but no `marker`. The `alt` gateway still has the kernel bound to `far.ipynb`:

In [ ]:
kid = re.search(r'^(\w+) ', await text('list_kernels', host=''), re.M).group(1)
test_eq(await text('exec', code='marker'), '7')
await text('use_kernel', kernel=kid, host='')
assert 'NameError' in await text('exec', code='marker')
test_eq(await text('exec', code='base'), '42')
await text('list_kernels', host='alt')

'c722209268df41aca150ce6477304a7c  alive  kernel=ipymini  language=python  connections=1  dlgname=far.ipynb  <- current'

Rustygate returns this PIL image as a text block followed by an MCP image block. It follows `fastcore.nbio.IMG_MIMES` when choosing a format. PIL supplies PNG and JPEG representations, and that preference order selects JPEG. The router passes both content blocks to the client unchanged:


In [ ]:
r = await call('exec', code=r'''
from PIL import Image
Image.new("RGB", (300, 200), "red")
''')
blocks = r['content']
test_eq([b['type'] for b in blocks], ['text', 'image'])
test_eq(blocks[1]['mimeType'], 'image/jpeg')
{k:v for k,v in blocks[1].items() if k != 'data'}


{'type': 'image', 'mimeType': 'image/jpeg'}

`create(kernel='luau')` selects a Luau kernel on the chosen gateway. Luau doesn't run Python startup code. `exec` runs in the selected kernel without choosing a language. Requesting a different implementation for the same binding fails without changing the selection or state:


In [ ]:
created = await text('create', dlgname='native.ipynb', kernel='luau', host='alt')
assert 'language=luau' in created
test_eq(await text('exec', code='saved=42; return saved, base == nil'), '42\ttrue')
wrong = await call('create', dlgname='native.ipynb', kernel='py')
assert wrong['isError'] and 'not py' in txt(wrong)
test_eq(await text('exec', code='saved'), '42')
txt(wrong)

'kernel cab84350de774de7a0801d0628002018 uses implementation "luau", not ipymini; select a matching kernel with use_kernel or create'

`restart` clears the Luau kernel's state, including `saved`. It still doesn't run Python startup, which would define `base`. Switching back to the original Python kernel reveals its unchanged state:

In [ ]:
await text('restart')
test_eq(await text('exec', code='return saved, base'), 'nil\tnil')
await text('use_kernel', kernel=kid, host='')
base = await text('exec', code='base')
test_eq(base, '42')
base

'42'

## Closing sessions

`aclose` attempts every session close and stops an owned child gateway even if a close fails. It raises the session errors together after cleanup.

In [ ]:
#| export
@patch
async def aclose(self:Router):
    "Close all sessions and their autoclose kernels, then stop any owned gateway."
    try: results = await asyncio.gather(*(s.aclose() for s in self.sessions.values()), return_exceptions=True)
    finally:
        if self.child: self.child.stop()
    errors = [r for r in results if isinstance(r, BaseException)]
    if errors: raise BaseExceptionGroup('MCP session cleanup failed', errors)

Ending the router's sessions stops the kernels it created with autoclose. Selecting an existing kernel does not make the router responsible for its lifetime.

Our gateways were already running when the router connected. It owns neither process. A new router can still connect to both gateways, but neither has any remaining kernels:

In [ ]:
await router.aclose()
assert router.child is None
chk = Router(cfgd)
near = await (await chk.session()).text('list_kernels')
far = await (await chk.session('alt')).text('list_kernels')
await chk.aclose()
test_eq((near, far), ('no kernels', 'no kernels'))
near, far


('no kernels', 'no kernels')

## The stdio server

`main` serves one router over stdio. The router opens each gateway session on first use. Initialization opens the local session, starting a child gateway if necessary. A later `tools/list` request fetches the tool schemas.

On exit, `main` closes all gateway sessions and stops any child gateway it owns. Stopping that gateway also stops its kernels. `--quiet` suppresses kernel startup output in replies. `--cfgdir` names the directory holding `startup.py`, `inspectors.py`, and `gateways.toml`, in place of `~/.config/clikernel/`.


In [ ]:
#| export
@call_parse
def main(
    quiet:store_true=False,  # Keep startup output out of replies
    cfgdir:str=None,  # Directory holding `startup.py`, `inspectors.py`, and `gateways.toml`; `~/.config/clikernel/` if unset
):
    "Run `clikernel-mcp` as a stdio server."
    async def _main():
        router = Router(cfgdir, quiet=quiet)
        try: await serve_stdio(router)
        finally: await router.aclose()
    asyncio.run(_main())


We can also use the installed `clikernel-mcp` command through an MCP stdio client. Call `create` before `exec`. `--cfgdir` points it at the temporary config directory, so the Python kernel it creates has `base` from that `startup.py`:

In [ ]:
import shutil
from mcpmini.core import MCPClient


In [ ]:
#| hide
cmd = shutil.which('clikernel-mcp')
assert cmd, 'clikernel-mcp script not installed'
env = os.environ | {'CLIKERNEL_HOST': g.url, 'XDG_CONFIG_HOME': str(cfgd)}

In [ ]:
async with MCPClient.stdio([cmd, '--cfgdir', str(cfgd)], env=env) as m:
    assert 'exec' in dir(m.tools) and 'py' not in dir(m.tools) and 'lua' not in dir(m.tools)
    await m.tools.create(kernel='py')
    res = await m.tools.exec(code='base')
    await m.tools.create(dlgname='stdio-native.ipynb', kernel='luau')
    native_res = await m.tools.exec(code='6*7')
test_eq((res, native_res), ('42', '42'))
{'Python': res, 'Luau': native_res}


{'Python': '42', 'Luau': '42'}

## A live client

These optional lessons use Claude Code's headless Agent SDK. They spend model tokens and have `#| eval: false` to exclude them from automated runs. Run them manually after changing the tools or updating Claude Code.

Shared setup restricts Claude to `create` and `exec`. The `ask_claude` helper returns its final answer and the tool names it used.


In [ ]:
#| eval: false
import logging, random
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ToolUseBlock, ResultMessage

In [ ]:
#| hide
#| eval: false
logging.getLogger('claude_agent_sdk').setLevel(logging.WARNING)
live_tmp = tempfile.TemporaryDirectory()
live_dir = Path(live_tmp.name)
lenv = os.environ | {'CLIKERNEL_HOST': g.url, 'XDG_CONFIG_HOME': live_tmp.name}
opts = ClaudeAgentOptions(mcp_servers=dict(ck=dict(type='stdio', command=cmd, env=lenv)), cwd=live_tmp.name,
    allowed_tools=['mcp__ck__create', 'mcp__ck__exec'], max_turns=6)

async def ask_claude(prompt):
    msgs = [m async for m in query(prompt=prompt, options=opts)]
    used = {b.name for m in msgs if isinstance(m, AssistantMessage) for b in m.content if isinstance(b, ToolUseBlock)}
    return first(m.result for m in msgs if isinstance(m, ResultMessage)), used

Claude must create a `py` kernel and use it to compute the result:

In [ ]:
#| eval: false
prompt = r'''Using the ck MCP tools, create a py kernel, then exec 17*19. Reply with just the number.'''
res, used = await ask_claude(prompt)
assert {'mcp__ck__create', 'mcp__ck__exec'} <= used
assert '323' in res
res


'323'

The next test checks image delivery through the router, stdio transport, and Claude Code's client. It chooses a random color without naming it in the prompt. The requested Python code reads the color from a file and displays a plain image. Claude must identify the color from that image. The final assertion compares its answer with the chosen color.

In [ ]:
#| eval: false
color = random.choice(['red', 'green', 'blue', 'yellow', 'purple', 'orange'])
(live_dir/'color.txt').write_text(color)
prompt = r'''
Using the ck MCP tools, create a py kernel, then run exactly this code with exec:

from PIL import Image
Image.new('RGB', (200,200), open('color.txt').read().strip())

Do not execute anything else or read color.txt any other way.
The exec result includes an image. Reply with just the color of that image.
'''
res2, _ = await ask_claude(prompt)
assert color in res2.lower()
color, res2

('purple', 'Purple.')

In [ ]:
#| hide
#| eval: false
live_tmp.cleanup()

In [ ]:
#| hide
g.stop()
g2.stop()
del os.environ['CLIKERNEL_HOST']
tmp.cleanup()


In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()